Energy Market Executive Analysis (2016 - 2026)
Automated Report generated via Dagster & dbt

This notebook provides a comprehensive analysis of the US energy landscape, integrating historical EIA data with synthetic environmental metrics. The following visualizations are designed to track market volatility, consumption growth, and the economic impact of the transition to renewable energy.

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# --- Metadata Tag: parameters ---
# Dagster will inject the database_path here
database_path = "database/energy_data.duckdb"

In [3]:
if not os.path.exists(database_path):
    database_path = os.path.abspath(os.path.join("..", database_path))

con = duckdb.connect(database_path)
df = con.execute("SELECT * FROM monthly_state_trends").df()
con.close()

print(f"Columns in DB: {df.columns.tolist()}")

# 1. Transform 'period' or 'month' into standardized date columns
if 'period' in df.columns:
    df['month'] = pd.to_datetime(df['period'])
    df['year'] = df['month'].dt.year
    df['month_num'] = df['month'].dt.month
    print("Standardized 'period' into 'year' and 'month'.")
elif 'month' in df.columns:
    df['month'] = pd.to_datetime(df['month'])
    df['year'] = df['month'].dt.year
    print("Standardized 'month' into 'year'.")
else:
    date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
    if date_col:
        df['month'] = pd.to_datetime(df[date_col[0]])
        df['year'] = df['month'].dt.year
    else:
        raise KeyError(f"No 'period' or 'year' column found. Available: {df.columns.tolist()}")

# 2. Ensure both variable names are ready for later cells
region_df = df.copy()

Columns in DB: ['period', 'state_code', 'state_name', 'census_region', 'market_type', 'price_cents_kwh', 'sales_mwh', 'carbon_intensity', 'weather_index', 'renewable_share', 'price_to_carbon_ratio']
Standardized 'period' into 'year' and 'month'.


1. 5-Year Price Forecast by Region
Objective: To project future energy costs based on the 10-year historical CAGR (Compound Annual Growth Rate).

This visualization combines Historical Data (solid lines) with Forecasted Data (dotted lines). By applying a linear regression to the past 10 years of regional pricing, we provide an automated estimate of where prices will land in 2031.

In [20]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# 1. Identify Columns
price_col = 'price_cents_kwh' if 'price_cents_kwh' in df.columns else 'price'
date_col = 'month' if 'month' in df.columns else 'period'

# Ensure date is datetime
df[date_col] = pd.to_datetime(df[date_col])

# 2. Setup Figure
regions = df['census_region'].unique()
colors = px.colors.qualitative.Bold
fig = go.Figure()

start_date = pd.to_datetime('2016-01-01')

for i, region in enumerate(regions):
    color = colors[i % len(colors)]
    
    # Process History (2016-2026)
    reg_hist = df[(df['census_region'] == region) & (df[date_col] >= start_date)].copy()
    hist_monthly = reg_hist.groupby(date_col)[price_col].mean().reset_index().sort_values(date_col)
    
    if len(hist_monthly) < 2:
        continue
        
    # CAGR Calculation
    v_start = hist_monthly[price_col].iloc[0]
    v_end = hist_monthly[price_col].iloc[-1]
    years_diff = (hist_monthly[date_col].max() - hist_monthly[date_col].min()).days / 365.25
    cagr = (v_end / v_start) ** (1 / years_diff) - 1 if years_diff > 0 else 0
    
    # Generate Forecast (2026-2031)
    latest_date = hist_monthly[date_col].max()
    future_dates = pd.date_range(start=latest_date, periods=61, freq='MS')
    future_prices = [v_end * ((1 + cagr) ** (j / 12)) for j in range(len(future_dates))]
    
    # 3. Add Traces
    # Historical (Solid)
    fig.add_trace(go.Scatter(
        x=hist_monthly[date_col], y=hist_monthly[price_col],
        mode='lines', name=region,
        line=dict(color=color, width=3),
        legendgroup=region,
        hovertemplate="<b>" + region + "</b><br>%{x|%Y}: %{y:.2f}¢<extra></extra>"
    ))
    
    # Forecast (Dotted)
    fig.add_trace(go.Scatter(
        x=future_dates, y=future_prices,
        mode='lines', name=f"{region} Forecast",
        line=dict(color=color, width=2, dash='dot'),
        legendgroup=region,
        showlegend=False,
        hovertemplate="<b>" + region + " (Forecast)</b><br>%{x|%Y}: %{y:.2f}¢<extra></extra>"
    ))

# 4. FIXED SCALE: 0 to 35 cents
fig.update_layout(
    title="<b>Regional Price Evolution & 2031 Projection</b><br><sup>Focused Scale (0-25¢) to highlight regional volatility</sup>",
    xaxis_title="Year",
    yaxis_title="Price (¢/kWh)",
    template="plotly_white",
    hovermode="x unified",
    yaxis=dict(
        range=[0, 35],  # This fixes the scale as requested
        tickprefix="¢",
        dtick=5,        # Adds a grid line every 5 cents
        gridcolor='lightgrey'
    ),
    xaxis=dict(showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

2. Volumetric Consumption AnalysisObjective: To visualize total energy demand (load) and identify regional growth patterns.This stacked area chart displays the cumulative Sales in Megawatt-hours ($MWh$). It highlights the "energy footprint" of each region, making it easy to identify if certain regions are growing in consumption faster than others, which directly impacts grid stability and infrastructure requirements.

In [12]:
fig2 = px.area(
    region_df, 
    x="period", 
    y="sales_mwh", 
    color="census_region",
    title="Cumulative Energy Consumption (MWh) by Region (2016-2026)",
    labels={"sales_mwh": "Total Sales (MWh)"}
)
fig2.show()

3. The Renewable-Price Correlation Matrix
Objective: To investigate the economic impact of the green energy transition on consumer pricing.

This scatter plot maps the relationship between a state’s Renewable Share % and its Electricity Price.

In [74]:
fig3 = px.scatter(
    region_df, 
    x="renewable_share", 
    y="price_cents_kwh", 
    color="census_region",
    size="sales_mwh",
    hover_name="state_name",
    trendline="ols",
    title="Correlation: Renewable Adoption vs. Market Pricing",
    labels={"renewable_share": "Renewable Share %", "price_cents_kwh": "Price (¢/kWh)"}
)
fig3.show()

4. National Carbon Intensity Baseline
Objective: A geographic "Heatmap" of carbon emissions related to energy production.

This choropleth map ranks states by their Carbon Intensity Index. Using a divergent color scale (Red to Green), it provides an immediate visual audit of "Carbon Hotspots" versus "Green Leaders," offering a high-level view of the environmental performance of the national power grid.

In [ ]:
max_year = df['year'].max()
map_data = df[df['year'] == max_year].groupby('state_code')['carbon_intensity'].mean().reset_index()

fig4 = px.choropleth(
    map_data,
    locations='state_code',
    locationmode="USA-states",
    color='carbon_intensity',
    scope="usa",
    color_continuous_scale="RdYlGn_r",
    title=f"{max_year} Average Carbon Intensity by State",
    labels={'carbon_intensity': 'Carbon Index'}
)
fig4.show()

 Regional Market Share & Price ComparisonObjective: To compare total energy sales volume ($MWh$) against average pricing ($¢/kWh$) across regions.This visualization uses a Sunburst Chart. It allows you to see the hierarchy of the US energy market: the inner ring shows the Census Regions, and the outer ring shows the States within them. The size of each slice represents the Sales Volume, while the color represents the Average Price

In [ ]:
sunburst_df = region_df.groupby(['census_region', 'state_code']).agg({
    'sales_mwh': 'sum',
    'price_cents_kwh': 'mean'
}).reset_index()

fig5 = px.sunburst(
    sunburst_df, 
    path=['census_region', 'state_code'], 
    values='sales_mwh',
    color='price_cents_kwh', 
    color_continuous_scale='RdYlGn_r',
    title="Regional Market Share (Size) vs. Price (Color)",
    labels={'price_cents_kwh': 'Avg Price', 'sales_mwh': 'Total MWh'}
)

fig5.update_layout(margin=dict(t=40, l=0, r=0, b=0))
fig5.show()